# 🎯 MNIST Handwritten Digit Recognition
### Advanced Deep Learning Project — Task 2 (Machine Learning Internship)

**Arch Technologies — Machine Learning Internship, Month 1**

---

**Project Goal:** Build, train, evaluate, and compare multiple machine learning models to classify handwritten digits (0–9) from the MNIST dataset, using a full deep-learning pipeline with professional visualizations and interactive analytics.

**Pipeline Overview:**
1. 📥 Data Loading & Inspection
2. 🔍 Exploratory Data Analysis (EDA) — interactive charts
3. 🧹 Preprocessing & Augmentation
4. 🧠 Model Architecture (Advanced CNN)
5. 🚀 Training with Callbacks
6. 📊 Evaluation — Confusion Matrix, ROC, Per-Class Metrics
7. ⚖️ Model Comparison (CNN vs DNN vs SVM vs Random Forest)
8. 💾 Save Artifacts for Deployment

> Run every cell **top to bottom** (Runtime → Run all). GPU runtime recommended: `Runtime → Change runtime type → GPU`.


## 1️⃣ Setup & Library Imports

In [ ]:
# Install/upgrade libraries needed for interactive visualizations (Colab usually has these, this just ensures versions)
!pip install -q plotly scikit-learn seaborn --upgrade


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import pickle, os, time, warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)
print("✅ All libraries imported successfully!")


## 2️⃣ Load MNIST Dataset

The MNIST dataset contains **70,000 grayscale images** (28×28 pixels) of handwritten digits 0–9, split into 60,000 training and 10,000 testing samples.


In [ ]:
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

print("="*60)
print("📊 DATASET SHAPE")
print("="*60)
print(f"Training images : {X_train_raw.shape}")
print(f"Training labels : {y_train_raw.shape}")
print(f"Testing images  : {X_test_raw.shape}")
print(f"Testing labels  : {y_test_raw.shape}")
print(f"Pixel value range: {X_train_raw.min()} - {X_train_raw.max()}")


In [ ]:
# Summary statistics table
summary_df = pd.DataFrame({
    'Metric': ['Total Images', 'Training Images', 'Testing Images', 'Image Height', 'Image Width',
               'Number of Classes', 'Pixel Min', 'Pixel Max', 'Pixel Mean', 'Pixel Std Dev'],
    'Value': [
        X_train_raw.shape[0] + X_test_raw.shape[0],
        X_train_raw.shape[0],
        X_test_raw.shape[0],
        X_train_raw.shape[1],
        X_train_raw.shape[2],
        len(np.unique(y_train_raw)),
        X_train_raw.min(),
        X_train_raw.max(),
        round(X_train_raw.mean(), 2),
        round(X_train_raw.std(), 2)
    ]
})

fig = go.Figure(data=[go.Table(
    header=dict(values=['<b>Metric</b>', '<b>Value</b>'],
                fill_color='#667eea', font=dict(color='white', size=13), align='left', height=32),
    cells=dict(values=[summary_df.Metric, summary_df.Value],
               fill_color=[['#f4f4fb', 'white']*5], align='left', height=28)
)])
fig.update_layout(title='📋 Dataset Summary Statistics', height=420, margin=dict(t=50, b=10))
fig.show()


## 3️⃣ Exploratory Data Analysis (EDA)

### 3.1 Class Distribution
Checking whether the dataset is balanced across all 10 digit classes.


In [ ]:
train_counts = pd.Series(y_train_raw).value_counts().sort_index()
test_counts = pd.Series(y_test_raw).value_counts().sort_index()

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'bar'}, {'type': 'domain'}]],
                     subplot_titles=('Training Set — Count per Digit', 'Training Set — Proportion'))

fig.add_trace(go.Bar(x=train_counts.index, y=train_counts.values,
                      marker=dict(color=train_counts.values, colorscale='Viridis'),
                      text=train_counts.values, textposition='outside', name='Count'),
              row=1, col=1)

fig.add_trace(go.Pie(labels=[f'Digit {i}' for i in train_counts.index], values=train_counts.values,
                      hole=.35, textinfo='label+percent', showlegend=False), row=1, col=2)

fig.update_layout(height=450, title_text='🔢 Class Distribution — Training Set (60,000 images)',
                   showlegend=False, xaxis_title='Digit', yaxis_title='Count')
fig.show()

print("Training set is well balanced — each digit represents roughly 9-11% of samples.")


### 3.2 Sample Digits Grid
A random sample of digits from each class, rendered as an interactive image grid.

In [ ]:
fig = make_subplots(rows=2, cols=5, subplot_titles=[f'Digit: {i}' for i in range(10)])

for digit in range(10):
    idx = np.where(y_train_raw == digit)[0][0]
    row, col = (digit // 5) + 1, (digit % 5) + 1
    fig.add_trace(go.Heatmap(z=np.flipud(X_train_raw[idx]), colorscale='gray', showscale=False),
                  row=row, col=col)

fig.update_layout(height=500, title_text='🖼️ Sample Handwritten Digits (0–9)')
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()


### 3.3 Average Digit Images ('Digit Fingerprints')
Averaging all images per class reveals the typical stroke pattern for each digit.

In [ ]:
fig = make_subplots(rows=2, cols=5, subplot_titles=[f'Avg Digit: {i}' for i in range(10)])

for digit in range(10):
    avg_img = X_train_raw[y_train_raw == digit].mean(axis=0)
    row, col = (digit // 5) + 1, (digit % 5) + 1
    fig.add_trace(go.Heatmap(z=np.flipud(avg_img), colorscale='Viridis', showscale=False),
                  row=row, col=col)

fig.update_layout(height=500, title_text='📐 Average Pixel Intensity per Digit Class ("Digit Fingerprints")')
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()


### 3.4 Pixel Intensity Distribution
Comparing the distribution of non-zero pixel intensities across digit classes.

In [ ]:
intensity_data = []
for digit in range(10):
    imgs = X_train_raw[y_train_raw == digit][:500]
    nonzero_pixels = imgs[imgs > 15]
    intensity_data.append(nonzero_pixels)

fig = go.Figure()
for digit in range(10):
    fig.add_trace(go.Violin(y=intensity_data[digit], name=str(digit), box_visible=True,
                             meanline_visible=True, opacity=0.7))

fig.update_layout(title='🎻 Pixel Intensity Distribution by Digit (non-zero pixels)',
                   xaxis_title='Digit', yaxis_title='Pixel Intensity', height=450, showlegend=False)
fig.show()


### 3.5 Dimensionality Reduction — PCA Projection
Projecting the 784-dimensional pixel space down to 2D using PCA to visualize class separability.


In [ ]:
sample_size = 4000
sample_idx = np.random.choice(len(X_train_raw), sample_size, replace=False)
X_sample_flat = X_train_raw[sample_idx].reshape(sample_size, -1) / 255.0
y_sample = y_train_raw[sample_idx]

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sample_flat)

fig = px.scatter(x=X_pca[:, 0], y=X_pca[:, 1], color=y_sample.astype(str),
                  labels={'x': 'PCA Component 1', 'y': 'PCA Component 2', 'color': 'Digit'},
                  title=f'🗺️ PCA Projection of {sample_size} MNIST Images (2D)',
                  color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(height=550)
fig.show()

print(f"Explained variance by 2 PCA components: {pca.explained_variance_ratio_.sum()*100:.2f}%")


### 3.6 t-SNE Projection (Non-linear)
t-SNE captures non-linear structure and typically separates digit clusters more clearly than PCA. (Uses a smaller subsample for speed.)

In [ ]:
tsne_sample_size = 2000
tsne_idx = np.random.choice(len(X_train_raw), tsne_sample_size, replace=False)
X_tsne_flat = X_train_raw[tsne_idx].reshape(tsne_sample_size, -1) / 255.0
y_tsne = y_train_raw[tsne_idx]

print("Running t-SNE (this takes ~30-60 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_tsne_flat)
print("✅ t-SNE complete!")

fig = px.scatter(x=X_tsne[:, 0], y=X_tsne[:, 1], color=y_tsne.astype(str),
                  labels={'x': 't-SNE Dimension 1', 'y': 't-SNE Dimension 2', 'color': 'Digit'},
                  title=f'🌀 t-SNE Projection of {tsne_sample_size} MNIST Images',
                  color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_traces(marker=dict(size=6, opacity=0.75))
fig.update_layout(height=600)
fig.show()

print("Notice how t-SNE forms tight, well-separated clusters per digit — a strong signal that")
print("a well-designed classifier should achieve very high accuracy.")


## 4️⃣ Data Preprocessing

Steps:
1. Normalize pixel values to the range [0, 1]
2. Reshape to `(N, 28, 28, 1)` for CNN input
3. One-hot encode labels for categorical crossentropy
4. Create a held-out validation split


In [ ]:
X_train = (X_train_raw.astype('float32') / 255.0).reshape(-1, 28, 28, 1)
X_test = (X_test_raw.astype('float32') / 255.0).reshape(-1, 28, 28, 1)

y_train_cat = to_categorical(y_train_raw, 10)
y_test_cat = to_categorical(y_test_raw, 10)

# Flattened versions (needed for classical ML models later)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

print("✅ Preprocessing complete!")
print(f"CNN input shape   : {X_train.shape}")
print(f"Flattened shape   : {X_train_flat.shape}")
print(f"Labels (one-hot)  : {y_train_cat.shape}")


### 4.1 Data Augmentation Preview
Augmentation (small rotations, shifts, zooms) helps the model generalize to varied handwriting styles. Below is a preview of augmented versions of a single digit.

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

sample_digit = X_train[0:1]
aug_iter = datagen.flow(sample_digit, batch_size=1)

fig = make_subplots(rows=2, cols=5, subplot_titles=['Original'] + [f'Augmented {i}' for i in range(1, 10)])
images_to_show = [sample_digit[0].reshape(28, 28)] + [next(aug_iter)[0].reshape(28, 28) for _ in range(9)]

for i, img in enumerate(images_to_show):
    row, col = (i // 5) + 1, (i % 5) + 1
    fig.add_trace(go.Heatmap(z=np.flipud(img), colorscale='gray', showscale=False), row=row, col=col)

fig.update_layout(height=500, title_text='🔄 Data Augmentation Preview (rotation, shift, zoom)')
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()


## 5️⃣ Model Architecture — Advanced CNN

A 3-block Convolutional Neural Network with Batch Normalization and Dropout for regularization.


In [ ]:
def build_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    return model

model_cnn = build_cnn()
model_cnn.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                   loss='categorical_crossentropy', metrics=['accuracy'])
model_cnn.summary()


In [ ]:
# Visualize layer output shapes & parameter counts as an interactive table + bar chart
layer_info = []
for layer in model_cnn.layers:
    try:
        shape = layer.output_shape
    except:
        shape = 'N/A'
    layer_info.append({'Layer': layer.name, 'Type': layer.__class__.__name__,
                        'Output Shape': str(shape), 'Params': layer.count_params()})

layer_df = pd.DataFrame(layer_info)

fig = go.Figure(data=[go.Table(
    header=dict(values=['<b>Layer</b>', '<b>Type</b>', '<b>Output Shape</b>', '<b>Params</b>'],
                fill_color='#667eea', font=dict(color='white'), align='left'),
    cells=dict(values=[layer_df.Layer, layer_df.Type, layer_df['Output Shape'], layer_df.Params],
               fill_color=[['#f4f4fb','white']*len(layer_df)], align='left'))
])
fig.update_layout(title='🏗️ CNN Architecture — Layer Breakdown', height=650, margin=dict(t=50))
fig.show()

print(f"Total trainable parameters: {model_cnn.count_params():,}")


## 6️⃣ Model Training

Using **EarlyStopping** and **ReduceLROnPlateau** callbacks to avoid overfitting and fine-tune the learning rate automatically.


In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

print("🚀 Training CNN model...\n")
start_time = time.time()

history = model_cnn.fit(
    X_train, y_train_cat,
    epochs=20,
    batch_size=128,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time
print(f"\n✅ Training complete in {training_time:.1f} seconds!")


In [ ]:
# Interactive training curves
epochs_range = list(range(1, len(history.history['loss']) + 1))

fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss over Epochs', 'Accuracy over Epochs'))

fig.add_trace(go.Scatter(x=epochs_range, y=history.history['loss'], name='Train Loss',
                          mode='lines+markers', line=dict(color='#667eea', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_range, y=history.history['val_loss'], name='Val Loss',
                          mode='lines+markers', line=dict(color='#ef4444', width=3)), row=1, col=1)

fig.add_trace(go.Scatter(x=epochs_range, y=history.history['accuracy'], name='Train Accuracy',
                          mode='lines+markers', line=dict(color='#667eea', width=3)), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs_range, y=history.history['val_accuracy'], name='Val Accuracy',
                          mode='lines+markers', line=dict(color='#22c55e', width=3)), row=1, col=2)

fig.update_xaxes(title_text='Epoch')
fig.update_yaxes(title_text='Loss', row=1, col=1)
fig.update_yaxes(title_text='Accuracy', row=1, col=2)
fig.update_layout(height=450, title_text='📈 CNN Training History', legend=dict(orientation='h', y=-0.2))
fig.show()


## 7️⃣ Model Evaluation

### 7.1 Overall Test Set Performance


In [ ]:
y_pred_probs = model_cnn.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

accuracy = accuracy_score(y_test_raw, y_pred)
precision = precision_score(y_test_raw, y_pred, average='weighted')
recall = recall_score(y_test_raw, y_pred, average='weighted')
f1 = f1_score(y_test_raw, y_pred, average='weighted')

metrics_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Score': [accuracy, precision, recall, f1]
})

fig = go.Figure(go.Indicator(
    mode="gauge+number",
    value=accuracy * 100,
    title={'text': "Test Set Accuracy (%)"},
    gauge={'axis': {'range': [90, 100]},
           'bar': {'color': '#22c55e'},
           'steps': [{'range': [90, 95], 'color': '#fee2e2'},
                     {'range': [95, 98], 'color': '#fef9c3'},
                     {'range': [98, 100], 'color': '#dcfce7'}]}
))
fig.update_layout(height=400, title_text='🎯 CNN Model — Final Accuracy')
fig.show()

print(f"Accuracy:  {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")


### 7.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test_raw, y_pred)

fig = go.Figure(data=go.Heatmap(
    z=cm, x=[str(i) for i in range(10)], y=[str(i) for i in range(10)],
    colorscale='Blues', text=cm, texttemplate='%{text}', textfont={'size': 11},
    colorbar=dict(title='Count')
))
fig.update_layout(title='🔲 Confusion Matrix — CNN Model (Test Set)',
                   xaxis_title='Predicted Digit', yaxis_title='Actual Digit',
                   height=550, yaxis=dict(autorange='reversed'))
fig.show()


### 7.3 Per-Class Precision, Recall & F1-Score

In [ ]:
report_dict = classification_report(y_test_raw, y_pred, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose().iloc[:10].round(4)
report_df.index.name = 'Digit'
report_df = report_df.reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(x=report_df['Digit'], y=report_df['precision'], name='Precision'))
fig.add_trace(go.Bar(x=report_df['Digit'], y=report_df['recall'], name='Recall'))
fig.add_trace(go.Bar(x=report_df['Digit'], y=report_df['f1-score'], name='F1-Score'))

fig.update_layout(barmode='group', title='📊 Per-Digit Precision / Recall / F1-Score',
                   xaxis_title='Digit', yaxis_title='Score', height=450,
                   legend=dict(orientation='h', y=-0.2))
fig.show()

# Styled table
fig2 = go.Figure(data=[go.Table(
    header=dict(values=['<b>Digit</b>', '<b>Precision</b>', '<b>Recall</b>', '<b>F1-Score</b>', '<b>Support</b>'],
                fill_color='#667eea', font=dict(color='white'), align='center'),
    cells=dict(values=[report_df['Digit'], report_df['precision'], report_df['recall'],
                        report_df['f1-score'], report_df['support']],
               fill_color=[['#f4f4fb','white']*10], align='center'))
])
fig2.update_layout(title='📋 Classification Report (per digit)', height=420, margin=dict(t=50))
fig2.show()


### 7.4 ROC Curves (One-vs-Rest, Multiclass)

In [ ]:
y_test_bin = label_binarize(y_test_raw, classes=list(range(10)))

fig = go.Figure()
auc_scores = {}
colors = px.colors.qualitative.Bold

for i in range(10):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores[i] = roc_auc
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                              name=f'Digit {i} (AUC={roc_auc:.3f})',
                              line=dict(color=colors[i % len(colors)])))

fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Baseline',
                          line=dict(dash='dash', color='gray')))

fig.update_layout(title='📉 ROC Curves — One-vs-Rest per Digit',
                   xaxis_title='False Positive Rate', yaxis_title='True Positive Rate',
                   height=550, legend=dict(font=dict(size=10)))
fig.show()

print("Average AUC across all digits:", round(np.mean(list(auc_scores.values())), 4))


### 7.5 Misclassified Examples
Inspecting a sample of the model's mistakes helps understand failure modes.

In [ ]:
misclassified_idx = np.where(y_pred != y_test_raw)[0]
print(f"Total misclassified: {len(misclassified_idx)} out of {len(y_test_raw)} ({len(misclassified_idx)/len(y_test_raw)*100:.2f}%)")

sample_wrong = np.random.choice(misclassified_idx, min(10, len(misclassified_idx)), replace=False)

fig = make_subplots(rows=2, cols=5,
                     subplot_titles=[f'True:{y_test_raw[i]} Pred:{y_pred[i]}' for i in sample_wrong])

for pos, idx in enumerate(sample_wrong):
    row, col = (pos // 5) + 1, (pos % 5) + 1
    fig.add_trace(go.Heatmap(z=np.flipud(X_test_raw[idx]), colorscale='gray', showscale=False),
                  row=row, col=col)

fig.update_layout(height=500, title_text='❌ Sample Misclassified Digits')
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()


### 7.6 CNN Feature Maps (What the Model "Sees")
Visualizing intermediate activations from the first convolutional layer — this shows which strokes, edges and curves the CNN has learned to detect on a real test digit.

In [ ]:
from tensorflow.keras.models import Model

conv_layer_names = [layer.name for layer in model_cnn.layers if 'conv2d' in layer.name][:1]
sample_img = X_test[0:1]

activation_model = Model(inputs=model_cnn.inputs, outputs=[model_cnn.get_layer(name).output for name in conv_layer_names])
activations = activation_model.predict(sample_img, verbose=0)

first_layer_activation = activations[0]
n_filters = min(8, first_layer_activation.shape[-1])

fig = make_subplots(rows=1, cols=n_filters, subplot_titles=[f'Filter {i+1}' for i in range(n_filters)])
for i in range(n_filters):
    fig.add_trace(go.Heatmap(z=np.flipud(first_layer_activation[0, :, :, i]), colorscale='Viridis', showscale=False), row=1, col=i + 1)

fig.update_layout(height=280, title_text=f'🔬 Feature Maps — Layer "{conv_layer_names[0]}" (first {n_filters} filters, on a sample test digit)')
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()


### 7.7 Prediction Confidence Distribution
Comparing how confident the model is (max softmax probability) on correct vs incorrect predictions — a well-calibrated model should be noticeably more confident on the predictions it gets right.

In [ ]:
max_probs = y_pred_probs.max(axis=1)
correct_mask = y_pred == y_test_raw

fig = go.Figure()
fig.add_trace(go.Histogram(x=max_probs[correct_mask], name='Correct Predictions',
                            marker_color='#22c55e', opacity=0.75, nbinsx=40))
fig.add_trace(go.Histogram(x=max_probs[~correct_mask], name='Incorrect Predictions',
                            marker_color='#ef4444', opacity=0.75, nbinsx=40))

fig.update_layout(barmode='overlay', title='🎯 Prediction Confidence Distribution — Correct vs Incorrect',
                   xaxis_title='Max Softmax Probability (Confidence)', yaxis_title='Count', height=450)
fig.show()

print(f"Average confidence on correct predictions   : {max_probs[correct_mask].mean():.4f}")
print(f"Average confidence on incorrect predictions : {max_probs[~correct_mask].mean():.4f}")


### 7.8 Precision–Recall Curves (One-vs-Rest)
Precision–Recall curves complement the ROC curves above and are especially informative for understanding the accuracy/coverage trade-off per digit class.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig = go.Figure()
ap_scores = {}
colors = px.colors.qualitative.Bold

for i in range(10):
    prec_i, rec_i, _ = precision_recall_curve(y_test_bin[:, i], y_pred_probs[:, i])
    ap = average_precision_score(y_test_bin[:, i], y_pred_probs[:, i])
    ap_scores[i] = ap
    fig.add_trace(go.Scatter(x=rec_i, y=prec_i, mode='lines',
                              name=f'Digit {i} (AP={ap:.3f})', line=dict(color=colors[i % len(colors)])))

fig.update_layout(title='📈 Precision–Recall Curves — One-vs-Rest per Digit',
                   xaxis_title='Recall', yaxis_title='Precision', height=550, legend=dict(font=dict(size=10)))
fig.show()

print("Average Precision (AP) across all digits:", round(np.mean(list(ap_scores.values())), 4))


## 8️⃣ Model Comparison

To validate that the CNN is genuinely the best choice, we train three additional baseline models on a representative subsample (for speed) and compare all four side-by-side.

| Model | Type | Notes |
|---|---|---|
| CNN | Deep Learning | Learns spatial features directly from pixels |
| Dense NN (MLP) | Deep Learning | Fully-connected, no spatial awareness |
| SVM (RBF kernel) | Classical ML | Strong baseline for image classification |
| Random Forest | Classical ML | Ensemble of decision trees |


In [ ]:
# Use a subsample for the classical models (SVM/RF can be slow on 60k samples)
sub_n = 8000
sub_idx = np.random.choice(len(X_train_flat), sub_n, replace=False)
X_sub, y_sub = X_train_flat[sub_idx], y_train_raw[sub_idx]

test_sub_n = 2000
test_sub_idx = np.random.choice(len(X_test_flat), test_sub_n, replace=False)
X_test_sub, y_test_sub = X_test_flat[test_sub_idx], y_test_raw[test_sub_idx]

results = {}

# --- Dense NN (MLP) ---
print("Training Dense Neural Network (MLP)...")
mlp = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=50, random_state=42, early_stopping=True)
mlp.fit(X_sub, y_sub)
pred_mlp = mlp.predict(X_test_sub)
results['Dense NN (MLP)'] = pred_mlp

# --- SVM ---
print("Training SVM (RBF kernel)...")
svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(X_sub, y_sub)
pred_svm = svm.predict(X_test_sub)
results['SVM'] = pred_svm

# --- Random Forest ---
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
rf.fit(X_sub, y_sub)
pred_rf = rf.predict(X_test_sub)
results['Random Forest'] = pred_rf

# --- CNN on the same test subsample (for fair comparison) ---
cnn_probs_sub = model_cnn.predict(X_test[test_sub_idx], verbose=0)
pred_cnn_sub = np.argmax(cnn_probs_sub, axis=1)
results['CNN'] = pred_cnn_sub

print("✅ All baseline models trained!")


In [ ]:
comparison_rows = []
for name, preds in results.items():
    acc = accuracy_score(y_test_sub, preds)
    prec = precision_score(y_test_sub, preds, average='weighted')
    rec = recall_score(y_test_sub, preds, average='weighted')
    f1s = f1_score(y_test_sub, preds, average='weighted')
    comparison_rows.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1s})

comparison_df = pd.DataFrame(comparison_rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)

fig = go.Figure()
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    fig.add_trace(go.Bar(x=comparison_df['Model'], y=comparison_df[metric], name=metric,
                          text=comparison_df[metric].apply(lambda v: f'{v*100:.1f}%'), textposition='outside'))

fig.update_layout(barmode='group', title='⚖️ Model Comparison — All Metrics (evaluated on same test subsample)',
                   yaxis_title='Score', height=500, legend=dict(orientation='h', y=-0.2))
fig.show()

fig2 = go.Figure(data=[go.Table(
    header=dict(values=[f'<b>{c}</b>' for c in comparison_df.columns],
                fill_color='#667eea', font=dict(color='white'), align='center'),
    cells=dict(values=[comparison_df[c].apply(lambda v: f'{v:.4f}' if isinstance(v, float) else v)
                        for c in comparison_df.columns],
               fill_color=[['#dcfce7' if i == 0 else ('#f4f4fb' if i % 2 else 'white')
                            for i in range(len(comparison_df))]] * len(comparison_df.columns),
               align='center'))
])
fig2.update_layout(title='🏆 Final Model Comparison Table (ranked by accuracy)', height=250, margin=dict(t=50))
fig2.show()

print(f"\n🏆 BEST MODEL: {comparison_df.iloc[0]['Model']} with {comparison_df.iloc[0]['Accuracy']*100:.2f}% accuracy")


### 8.1 Model Comparison — Radar Chart
A radar/polar view makes it easy to see, at a glance, which model is the most balanced across all four metrics — not just which one wins on accuracy alone.

In [ ]:
categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

fig = go.Figure()
for _, row in comparison_df.iterrows():
    fig.add_trace(go.Scatterpolar(r=[row[c] for c in categories] + [row[categories[0]]],
                                   theta=categories + [categories[0]],
                                   fill='toself', name=row['Model']))

fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
                   title='🕸️ Model Comparison — Radar Chart (balanced view across metrics)',
                   height=550, showlegend=True)
fig.show()


## 9️⃣ Save Model Artifacts

Saving the trained CNN model and supporting metadata for use in the Streamlit deployment app.


In [ ]:
os.makedirs('models', exist_ok=True)

# Save CNN model
model_cnn.save('models/mnist_cnn_model.h5')
print("✅ Saved: models/mnist_cnn_model.h5")

# Save training history
with open('models/training_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)
print("✅ Saved: models/training_history.pkl")

# Save final metrics summary
final_metrics = {
    'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1_score': f1,
    'training_time_seconds': training_time,
    'total_params': model_cnn.count_params()
}
with open('models/final_metrics.pkl', 'wb') as f:
    pickle.dump(final_metrics, f)
print("✅ Saved: models/final_metrics.pkl")

comparison_df.to_csv('models/model_comparison.csv', index=False)
print("✅ Saved: models/model_comparison.csv")


In [ ]:
from google.colab import files

files.download('models/mnist_cnn_model.h5')
files.download('models/final_metrics.pkl')
files.download('models/model_comparison.csv')


### 9.1 Download Training History (.pkl) + All Artifacts (Zipped)
Downloading `training_history.pkl` directly (needed if you want to re-plot the training curves later in your UI app), plus a single zipped archive of everything in `models/` so it's easy to pull into your Streamlit/UI project in one go.

In [ ]:
import shutil

files.download('models/training_history.pkl')

shutil.make_archive('mnist_model_artifacts', 'zip', 'models')
print("✅ Created: mnist_model_artifacts.zip")

files.download('mnist_model_artifacts.zip')


## 🎉 Summary

| Item | Result |
|---|---|
| **Best Model** | Convolutional Neural Network (CNN) |
| **Test Accuracy** | ~99% |
| **Precision / Recall / F1** | ~99% each |
| **Dataset** | MNIST — 70,000 images, 10 balanced classes |
| **Architecture** | 3 Conv blocks + BatchNorm + Dropout + Dense head |
| **Training Time** | ~1–2 minutes on GPU |
| **Artifacts Saved** | `mnist_cnn_model.h5`, training history, metrics, comparison table |

**Next step:** Load `mnist_cnn_model.h5` into the companion Streamlit app for interactive, real-time digit recognition (draw or upload an image).
